In [1]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [2]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np
import networkx as nx

In [4]:
# define a function to load a pickle
def load_pickle(file_name):
    if os.path.exists(file_name):
        with open(file_name, 'rb') as handle:
            de_pickle = pickle.load(handle)
    else:
        de_pickle = None
        print("file does not exist")
    return de_pickle

In [5]:
char_matrix = load_pickle(file_name = 'char_matrix.pkl')
letter_dict = load_pickle(file_name = 'letter_dict.pkl')
letter_rank_dict = load_pickle(file_name ='letter_rank_dict.pkl')
word_df = load_pickle(file_name = 'word_df.pkl')

C:\Users\babbm\AppData\Local\Temp\ipykernel_46572\773985408.py:5: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  de_pickle = pickle.load(handle)


In [6]:
word_df.head()

,word,lcase,word_id,n_unique_chars,word_score,word_group
0,aalii,aalii,0,3,20449.0,-2134795976539657997
1,Aaron,aaron,1,4,20642.0,5168226787426640364
2,abaca,abaca,2,3,20009.0,7298090979578241273
3,aback,aback,3,4,15456.0,6926502400987240042
4,abaff,abaff,4,3,14038.0,7931255010571806167


In [7]:
word_df.shape

(9972, 6)

In [8]:
# using the word_group, select entries
word_df = word_df.drop_duplicates(subset = ['word_group'])

In [9]:
word_df = word_df.loc[word_df['n_unique_chars'] == 5, :].sort_values(by = 'lcase').reset_index(drop = True)

In [10]:
word_df.head()

,word,lcase,word_id,n_unique_chars,word_score,word_group
0,abhor,abhor,22,5,15210.0,-6309929865134252395
1,abide,abide,23,5,16642.0,-3047576767538267104
2,Abies,abies,25,5,17816.0,7894952363982762409
3,abilo,abilo,26,5,16157.0,-4762514508553607400
4,abler,abler,28,5,17943.0,-7669602973530280345


In [11]:
word_df['word_id'] = range(0, word_df.shape[0])

In [12]:
word_id_list = word_df['word_id'].to_numpy(dtype = np.int16)

In [13]:
word_id_list

array([   0,    1,    2, ..., 4350, 4351, 4352],
      shape=(4353,), dtype=int16)

In [14]:
word_df['lcase'].map(lambda x: len(set(x))).describe()

count    4353.0
mean        5.0
std         0.0
min         5.0
25%         5.0
50%         5.0
75%         5.0
max         5.0
Name: lcase, dtype: float64

In [15]:
# build a char_matrix
char_matrix = np.zeros(shape = (word_df.shape[0], 26), dtype = np.int8)
def build_char_matrix(row):
    word = sorted(row['lcase'])
    for iw, w in enumerate(word):
        char_matrix[row['word_id'], letter_dict[w]] += 1


In [16]:
outcome = word_df.apply(build_char_matrix, axis = 1)

In [17]:
char_matrix

array([[1, 1, 0, ..., 0, 0, 0],
       [1, 1, 0, ..., 0, 0, 0],
       [1, 1, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 1, 1],
       [0, 0, 1, ..., 0, 1, 1],
       [0, 0, 0, ..., 0, 1, 1]], shape=(4353, 26), dtype=int8)

In [18]:
char_matrix.max()

np.int8(1)

In [19]:
assert word_df.shape[0] == char_matrix.shape[0]

In [20]:
char_matrix.shape

(4353, 26)

In [21]:
# a solution
# vibex glyph muntz dwarf jocks

In [22]:
word_df['lcase_set'] = word_df['lcase'].map(lambda x: set(x))
word_df['lcase_tuple'] = word_df['lcase_set'].map(lambda x: tuple(x))

In [23]:
# build dictionaries
word_dict = {}
for i_row, my_row in word_df.iterrows():
    word_dict[my_row['lcase']] = my_row['lcase_set']

In [24]:
lc_set = set(ascii_lowercase)

In [25]:
vowel_set = set('aeiouy')

In [26]:
edge_list = []
for lc0, lc1 in combinations(word_df['lcase'], 2):
    w1w2_set = word_dict[lc0].union(word_dict[lc1])    
    if len(w1w2_set) == 10:

        # compute the number of vowels left over
        remaining_vowels = vowel_set.difference(w1w2_set)
        # need at least three vowels - for three separate words to continue
        if len(remaining_vowels) >= 3:
            
            existing_letters = ''.join(sorted(w1w2_set))
            remainder_letters = ''.join(sorted(lc_set.difference(existing_letters)))

            ##edge_list.append([ms[0], ms[1]])
            edge_list.append([lc0, lc1, existing_letters, remainder_letters ])
            #myg.add_edge(u_of_edge=lc0, v_of_edge=lc1)

In [27]:
# export to an edgelist
output = pd.DataFrame(data = edge_list, columns = ['w1', 'w2', 'l2_existing', 'r2_remainder'])

In [28]:
output.shape

(502235, 4)

In [29]:
output.head()

,w1,w2,l2_existing,r2_remainder
0,abhor,cleft,abcefhlort,dgijkmnpqsuvwxyz
1,abhor,clift,abcfhilort,degjkmnpqsuvwxyz
2,abhor,cling,abcghilnor,defjkmpqstuvwxyz
3,abhor,clink,abchiklnor,defgjmpqstuvwxyz
4,abhor,clint,abchilnort,defgjkmpqsuvwxyz


In [30]:
remainder = output.drop_duplicates(subset = ['l2_existing', 'r2_remainder'])

In [31]:
remainder.shape

(185268, 4)

In [32]:
remainder.head()

,w1,w2,l2_existing,r2_remainder
0,abhor,cleft,abcefhlort,dgijkmnpqsuvwxyz
1,abhor,clift,abcfhilort,degjkmnpqsuvwxyz
2,abhor,cling,abcghilnor,defjkmpqstuvwxyz
3,abhor,clink,abchiklnor,defgjmpqstuvwxyz
4,abhor,clint,abchilnort,defgjkmpqsuvwxyz


In [33]:
test_df = remainder.iloc[:1000]

In [34]:
working_word_df = word_df[['word_id', 'lcase']].copy()

In [35]:
def get_vowels(letters, r_word, min_vowel_count, existing_words):    

    # combine the l1 and the lcase to make l2
    letter_group = set(letters + r_word)
    rg_vowels = vowel_set.difference(letter_group)
    rg_vowel_count = len(rg_vowels)    

    if rg_vowel_count >= min_vowel_count:        
        letter_group = ''.join(sorted(letter_group))    
        remainder_group  = lc_set.difference(letter_group)
        remainder_group = ''.join(sorted(remainder_group))
        temp_list = [r_word, letter_group, remainder_group]
        existing_words.extend(temp_list)
        return existing_words
    else:
        return None

In [36]:
working_word_df = word_df[['word_id', 'lcase']].copy()

In [37]:
l3_df_list = []
for i_row, row in remainder.iterrows():
    
    w1, w2, l2, r2 = row
    # existing letters
    existing_id_list = []
    for l in l2:
        l_idx = letter_dict[l]   
        test_char_matrix = char_matrix[:, l_idx] == 1
        # print(word_id_list[test_char_matrix])
        existing_id_list.append(word_id_list[test_char_matrix])

    #print(inclusive_id_list)
    existing_id_list = np.concatenate(existing_id_list)
    

    remainder_id_list = np.setdiff1d(word_id_list, existing_id_list)
    if remainder_id_list.size > 0:                
    
        # do this later - winnow down the set list sooner
        r3_word_list = working_word_df.loc[working_word_df['word_id'].isin(remainder_id_list), 'lcase'].tolist()
        # print(r3_word_list)
        for r3_word in r3_word_list:
            output = get_vowels(letters=l2, r_word = r3_word, min_vowel_count=2, existing_words=[w1, w2])
            if output:
                l3_df_list.append(output)



In [38]:
l3_df = pd.DataFrame(data = l3_df_list, columns = ['w1', 'w2', 'w3', 'l3', 'r3'])

In [39]:
l3_df.head()

,w1,w2,w3,l3,r3
0,abhor,cleft,dimps,abcdefhilmoprst,gjknquvwxyz
1,abhor,cleft,jinks,abcefhijklnorst,dgmpquvwxyz
2,abhor,cleft,skimp,abcefhiklmoprst,dgjnquvwxyz
3,abhor,cleft,spink,abcefhiklnoprst,dgjmquvwxyz
4,abhor,cleft,spung,abcefghlnoprstu,dijkmqvwxyz


In [40]:
l3_df.shape

(4950787, 5)

In [41]:
test_df = l3_df[['w1', 'w2', 'w3', 'l3', 'r3']].drop_duplicates(subset = ['l3', 'r3'] )

In [42]:
test_df.shape

(233091, 5)

In [43]:
test_df.head()

,w1,w2,w3,l3,r3
0,abhor,cleft,dimps,abcdefhilmoprst,gjknquvwxyz
1,abhor,cleft,jinks,abcefhijklnorst,dgmpquvwxyz
2,abhor,cleft,skimp,abcefhiklmoprst,dgjnquvwxyz
3,abhor,cleft,spink,abcefhiklnoprst,dgjmquvwxyz
4,abhor,cleft,spung,abcefghlnoprstu,dijkmqvwxyz


In [44]:
level3_list = []
l4_df_list = []
for i_row, row in test_df.iterrows():

    w1, w2, w3, l3, r3 = row

    # existing letters
    existing_id_list = []
    for l in l3:
        l_idx = letter_dict[l]   
        test_char_matrix = char_matrix[:, l_idx] == 1
        # print(word_id_list[test_char_matrix])
        existing_id_list.append(word_id_list[test_char_matrix])

    #print(inclusive_id_list)
    existing_id_list = np.concatenate(existing_id_list)        
    
    remainder_id_list = np.setdiff1d(word_id_list, existing_id_list)
    if remainder_id_list.size > 0:                
        #print(l1, r1, outcome_ids.shape)        
        #level2_list.append([l1, r1, existing_id_list.shape[0], remainder_id_list.shape[0]])                       
        #         
        # l1
        # do this later - winnow down the set list sooner
        # do this later - winnow down the set list sooner
        r4_word_list = working_word_df.loc[working_word_df['word_id'].isin(remainder_id_list), 'lcase'].tolist()
        # print(r3_word_list)
        for r4_word in r4_word_list:
            output = get_vowels(letters=l3, r_word = r4_word, min_vowel_count=1, existing_words=[w1, w2, w3])
            if output:
                l4_df_list.append(output)
        
                
                



In [45]:
l4_df = pd.DataFrame(data = l4_df_list, columns =  ['w1', 'w2', 'w3', 'w4', 'l4', 'r4'])

In [49]:
l4_df.shape

(231503, 6)

In [50]:
l4_df.head()

,w1,w2,w3,w4,l4,r4
0,abhor,fjeld,spung,twick,abcdefghijklnoprstuw,mqvxyz
1,abhor,fjeld,twick,spung,abcdefghijklnoprstuw,mqvxyz
2,abhor,spung,twick,fjeld,abcdefghijklnoprstuw,mqvxyz
3,abler,pfund,smock,wight,abcdefghiklmnoprstuw,jqvxyz
4,abler,pfund,wight,smock,abcdefghiklmnoprstuw,jqvxyz


In [51]:
test_df = l4_df.drop_duplicates(subset = ['l4', 'r4'])

In [52]:
test_df.shape

(6368, 6)

In [57]:
test_df.to_excel('level_4.xlsx', index = False)

In [53]:
l5_df_list = []
for i_row, row in test_df.iterrows():
    print(i_row)

    w1, w2, w3, w4, l4, r4 = row

    # existing letters
    existing_id_list = []
    for l in l4:
        l_idx = letter_dict[l]   
        test_char_matrix = char_matrix[:, l_idx] == 1
        # print(word_id_list[test_char_matrix])
        existing_id_list.append(word_id_list[test_char_matrix])

    #print(inclusive_id_list)
    existing_id_list = np.concatenate(existing_id_list)        
    
    remainder_id_list = np.setdiff1d(word_id_list, existing_id_list)
    if remainder_id_list.size > 0:                
        #print(l1, r1, outcome_ids.shape)        
        #level2_list.append([l1, r1, existing_id_list.shape[0], remainder_id_list.shape[0]])                       
        #         
        # l1
        # do this later - winnow down the set list sooner
        # do this later - winnow down the set list sooner
        r5_word_list = working_word_df.loc[working_word_df['word_id'].isin(remainder_id_list), 'lcase'].tolist()
        # print(r3_word_list)
        for r5_word in r5_word_list:
            output = get_vowels(letters=l4, r_word = r5_word, min_vowel_count=0, existing_words=[w1, w2, w3, w4])
            if output:
                l5_df_list.append(output)
        
                
                



0
3
6
9
13
15
19
20
44
47
50
57
60
63
69
77
88
90
91
92
100
111
112
113
115
116
121
123
124
126
156
210
240
250
252
255
257
261
266
269
270
300
351
353
370
371
373
375
379
381
383
391
393
415
438
443
446
447
449
456
462
463
504
511
520
560
570
571
572
576
583
584
592
598
599
601
602
603
611
618
642
643
663
668
669
670
671
673
679
681
682
685
686
687
692
696
727
741
742
745
746
749
767
772
773
782
784
786
795
797
800
822
825
826
828
829
830
832
833
834
835
836
837
840
842
843
844
848
852
853
855
857
885
886
888
905
920
929
939
940
965
966
986
1002
1039
1054
1081
1090
1101
1134
1136
1187
1193
1194
1195
1208
1213
1225
1231
1260
1261
1272
1273
1274
1275
1276
1278
1301
1359
1538
1557
1682
1691
1692
1701
1704
1711
1713
1738
1742
1744
1753
1754
1756
1773
1815
1816
1818
1843
1847
1884
1901
1920
1921
1923
1949
1951
1952
1967
2064
2065
2066
2103
2133
2135
2174
2199
2236
2264
2349
2355
2374
2448
2456
2482
2529
2539
2541
2548
2555
2582
2610
2614
2636
2651
2693
2700
2703
2705
2710
2735
2769
2781
27

In [54]:
l5_df = pd.DataFrame(data = l5_df_list, columns =  ['w1', 'w2', 'w3', 'w4', 'w5', 'l5', 'r5'])

In [56]:
l5_df.head()

,w1,w2,w3,w4,w5,l5,r5


In [ ]:
l2_words['l2'].unique().shape

In [ ]:
sc_df['inclusive_diff'] = sc_df['inclusive'] + sc_df['outcome']

In [ ]:
sc_df

In [ ]:
sc_df['inclusive_diff'].describe()

In [ ]:
np.unique(inclusive_id_list).shape

In [ ]:
np.unique(remainder_id_list).shape

In [ ]:
outcome_ids = np.setdiff1d(remainder_id_list, inclusive_id_list)

In [ ]:
outcome_ids

In [ ]:
char_index_vector = [letter_dict[x] for x in l1]

In [ ]:
char_index_vector

In [ ]:
char_matrix.sum()

In [ ]:
for x in lett

In [ ]:
testo = char_matrix[:, char_index_vector].sum(axis = 1) > 0

In [ ]:
testo.sum()

In [ ]:
testo.shape

In [ ]:
def create_1d_vector(letters):
    output = np.zeros((26,), dtype = np.int8)
    for l in letters:
        output[letter_dict[l]] = 1
    return output



In [ ]:
l1_v = create_1d_vector(letters = remainder['letters'].iloc[0])
l1_v

In [ ]:
outcome = (char_matrix + l1_v)

In [ ]:
outcome

In [ ]:
(outcome.max(axis = 1) == 1).sum()

In [ ]:
remainder_word_dict = {}
# find possible words
def get_remainder_words(row):
    
    # existing letters
    inclusive_id_list = []
    for l in row['letters']:
        l_idx = letter_dict[l]        
        inclusive_id_list.append(word_id_list[char_matrix[:, l_idx] == 0])

    #print(inclusive_id_list)
    inclusive_id_list = np.concatenate(inclusive_id_list)

    # remainder letters            
    remainder_id_list = []
    for l in row['remainder']:
        l_idx = letter_dict[l]
        remainder_id_list.append(word_id_list[char_matrix[:, l_idx] == 0])
    
    remainder_id_list = np.concatenate(remainder_id_list)
    
    outcome_ids = np.setdiff1d(remainder_id_list, inclusive_id_list)
    return outcome_ids.shape[0]

In [ ]:
remainder.iloc[:2].apply(get_remainder_words, axis = 1)

In [ ]:
def convert_binary()

In [ ]:
# convert letters binary

In [ ]:
word_df['lid'] = word_df['lcase'].map(lambda x: int(''.join([letter_dict[w] for x in x])))

In [ ]:
'bread' & 'beard'

In [ ]:
def add_that_shit(m1, m2):
    ps = m1[:, None, :] + m2[None, :, :]
    matching_pairs = np.argwhere(ps.max(axis = 2) == 1)
    matching_pairs = matching_pairs[:int(matching_pairs.shape[0] /  2) , :]       
    outcome = m1[matching_pairs[:, 0]] + m2[matching_pairs[:, 1]]
    return outcome, matching_pairs
    


In [ ]:
r1, mp1 = add_that_shit(m1 = char_matrix, m2 = char_matrix)

In [ ]:
r1.shape

In [ ]:
r_list = []
mp_list = []
chunker1 = np.arange(start = 0, stop = r1.shape[0] + 10000,step = 10000 )
for icc, curr_chunk in enumerate(chunker1[:-1]):    
    
    next_chunk = chunker1[icc + 1]
    print('l1', curr_chunk, next_chunk)
    r2, mp2 = add_that_shit(m1 = r1[curr_chunk:next_chunk, :], m2 = char_matrix)
    print('l1', r2.shape)
    r_list.append(r2)
    mp_list.append(mp2)


In [ ]:

    chunker2 = np.arange(start = 0, stop = r2.shape[0] + 10000,step = 10000 )
    for icc2, cc2 in enumerate(chunker2[:-1]):
        nc2 = chunker2[icc2 + 1]
        print('l2', cc2, nc2)
        r3, mp3 = add_that_shit(m1 = r2[cc2:nc2, :], m2 = char_matrix)
        print('l2', r3.shape)

        chunker3 = np.arange(start = 0, stop = r3.shape[0] + 10000,step = 10000)
        for icc3, cc3 in enumerate(chunker3[:-1]):
            nc3 = chunker3[icc3 + 1]
            print('l3', cc3, nc3)
            r4, mp4 = add_that_shit(m1 = r3[cc3:nc3, :], m2 = char_matrix)
            print('l4', r4.shape)
            if r4.size != 0:
                r_list.append([r1, r2, r3, r4])
                mp_list.append([mp1, mp2, mp3, mp4])
    


In [ ]:
word_df['lcase_set'] = word_df['lcase'].map(lambda x: set(x))
word_df['lcase_tuple'] = word_df['lcase_set'].map(lambda x: tuple(x))

In [ ]:
# build dictionaries
word_dict = {}
for i_row, my_row in word_df.iterrows():
    word_dict[my_row['lcase']] = my_row['lcase_set']

In [ ]:
r1 = add_that_shit(m1 = char_matrix, m2 = char_matrix)

In [ ]:
edge_list = []
for lc0, lc1 in combinations(word_df['lcase'], 2):
    if word_dict[lc0].isdisjoint(word_dict[lc1]):
        #edge_list.append([ms[0], ms[1]])
        myg.add_edge(u_of_edge=lc0, v_of_edge=lc1)


In [ ]:
import networkx as nx
from itertools import combinations

In [ ]:
myg = nx.Graph()
edge_list = []
for lc0, lc1 in combinations(word_df['lcase'], 2):
    if word_dict[lc0].isdisjoint(word_dict[lc1]):
        #edge_list.append([ms[0], ms[1]])
        myg.add_edge(u_of_edge=lc0, v_of_edge=lc1)


In [ ]:
for r0 in char_matrix:
    

In [ ]:
import scipy

In [ ]:
scipy.special.comb(word_df.shape[0], k = 5, exact=True)

In [ ]:
# chunks of 10K
chunker = range(0, r1.shape[0] + 10000, 10000)
list(chunker)


In [ ]:
# get the first words

In [ ]:
lm0 = char_matrix.copy()

In [ ]:
grand_output_list = []
for ii, l0 in enumerate(lm0[:10, :]):
    print(ii, l0)    
    outcome = ((lm0 + l0) <= 1).all(axis = 1)
    lm1 = lm0[outcome, :]
    for l1 in lm1:
        outcome = ((lm1 + l1) <= 1).all(axis = 1)
        lm2 = lm1[outcome, :]
        for l2 in lm2:
            outcome = ((lm2 + l2) <= 1).all(axis = 1)
            lm3 = lm2[outcome, :]
            for l3 in lm3:
                outcome = ((lm3 + l3) <= 1).all(axis = 1)
                lm4 = lm3[outcome, :]
                if lm4.shape[0] > 0:
                    grand_output_list.append([l0, l1, l2, l3, l4])

In [ ]:
ncm = char_matrix[outcome, :]
for irow in ncm:
    



In [ ]:
testo

In [ ]:
# thinking about this wrong... it's a case of letters left....

In [ ]:
word_df.head()

In [ ]:
word_df['lcase_set'] = word_df['lcase'].map(lambda x: set(x))
word_df['lcase_tuple'] = word_df['lcase_set'].map(lambda x: tuple(x))

In [ ]:
# build dictionaries
word_dict = {}
for i_row, my_row in word_df.iterrows():
    word_dict[my_row['lcase']] = my_row['lcase_set']

In [ ]:
import networkx as nx
from itertools import combinations

In [ ]:
myg = nx.Graph()
edge_list = []
for lc0, lc1 in combinations(word_df['lcase'], 2):
    if word_dict[lc0].isdisjoint(word_dict[lc1]):
        #edge_list.append([ms[0], ms[1]])
        myg.add_edge(u_of_edge=lc0, v_of_edge=lc1)


In [ ]:
myg.number_of_edges()

In [ ]:
word_df.head()

In [ ]:
nx.__version__

In [ ]:
#myg_backup = myg.copy()

In [ ]:
# get 100 words
test_list = word_df['lcase'].tolist()[:10]

In [ ]:
test_list

In [ ]:
#myg = myg_backup.copy()
output_list = []
for w0 in word_df['lcase']:    
#for w0 in test_list:
    reject_groupings = set()
    print(w0)
    l0 = [w0, '', '', '', '']
    #output_list.append(l0)        
    # this is immediately adjacent
    w1_set = set(nx.neighbors(G = myg, n = w0))
    fl0 = set(w0)    
    for w1 in w1_set:
        if reject_groupings.isdisjoint(permutations((w0, w1), r = 2)):
            if fl0.isdisjoint(set(w1)):
                fl1 = fl0.copy()
                fl1.update(w1)                 
                l1 = l0[:]
                l1[1] = w1
                #output_list.append(l1)                
                # immediately adjacent candidates        
                w2_set = set(nx.neighbors(G = myg, n = w1))
                w2_set = w2_set.intersection(w1_set)                
                if w2_set:
                    for w2 in w2_set:
                        if reject_groupings.isdisjoint(permutations((w0, w1,w2), r = 2)):
                            if fl1.isdisjoint(set(w2)):
                                fl2 = fl1.copy()
                                fl2.update(w2)
                                l2 = l1[:]
                                l2[2] = w2
                                #output_list.append(l2)                        
                                w3_set = set(nx.neighbors(G = myg, n = w2))                        
                                w3_set = w3_set.intersection(w2_set)                                
                                if w3_set:
                                    for w3 in w3_set:
                                        if reject_groupings.isdisjoint(permutations((w0, w1, w2, w3), r = 2)):                                        
                                            if fl2.isdisjoint(set(w3)):
                                                fl3 = fl2.copy()
                                                fl3.update(w3)
                                                l3 = l2[:]
                                                l3[3] = w3
                                                #output_list.append(l3)    
                                                # print(l3)
                                                w4_set = set(nx.neighbors(G = myg, n = w3))
                                                w4_set = w4_set.intersection(w3_set)
                                                if w4_set:
                                                    for w4 in w4_set:
                                                        if reject_groupings.isdisjoint(permutations((w0, w1, w2, w3, w4), r = 2)):
                                                            if fl3.isdisjoint(set(w4)):
                                                                fl4 = fl3.copy()
                                                                fl4.update(w4)
                                                                l4 = l3[:]
                                                                l4[4] = w4
                                                                output_list.append(l4)    
                                                                print(l4)
                                                else:                                                    
                                                    # compute combos                                                                                                    
                                                    #print(l3)
                                                    #print(w0, w1, w2, w3)
                                                    perms = list(permutations(l3, r = 2))
                                                    #print(perms)
                                                    reject_groupings.update(perms)
                                                    myg.remove_edges_from(perms)


#print('yay')  
#reject_groupings       

In [ ]:
output_list

In [ ]:
len(reject_groupings)

In [ ]:
myg.number_of_edges()

In [ ]:
output_list

In [ ]:
w3_seta

In [ ]:
reject_groupings

In [ ]:
len(testo)

In [ ]:
next_nodes

In [ ]:
myg.nodes(data = 'abhor')

In [ ]:
# this needs to be a recursive function


In [ ]:
test_outcome = np.where(tv1 == 0)[0]

In [ ]:
test_outcome

In [ ]:
outcome = char_matrix - tv1

In [ ]:
outcome

In [ ]:
outcome1 = outcome

In [ ]:
for wid in word_df['word_id'].tolist():
    cw = char_matrix[wid, :]
    
    # t/f list
    outcome = np.abs(char_matrix - cw).sum(1) == 10
    # next word id list
    wil2 = wil1[outcome]
    if wil2.shape[0] > 
    # next char matrix
    cm2 = cm1[outcome, :]
    
    
    # I'll need five different stages of the word_id_list 

    # 


    
    

In [ ]:
)

In [ ]:
tv1 = char_matrix[0, :]

In [ ]:
tv2 = char_matrix[-1, :]

In [ ]:
tv3 = tv2 - tv1

In [ ]:
testo = char_matrix - tv1

In [ ]:
testo

In [ ]:
testo2 = testo[np.abs(testo).sum(1) == 10, :]

In [ ]:
testo2

In [ ]:
# step 2
testo = testo[testo.sum(0)  ]

In [ ]:
# for each row in the char_matrix:
# substract from the chart_matrix:

In [ ]:
np.abs(tv3).sum()

In [ ]:
word_df.iloc[0]

In [ ]:
word_df.iloc[-1]